In [1]:
# ==================================================================================
# Colourization GAN for Full Size Images COCO Dataset — Single cell (production-ready)
# - WGAN-GP (critic), pretrain generator with L1 + perceptual (VGG)
# - Correct OpenCV LAB scaling, differentiable LAB->RGB for perceptual and metrics
# - Mixed-precision optional, robust checkpointing, TensorBoard logging
# ==================================================================================

import os
import random
import time
import shutil
import csv
import math
import gc
from pathlib import Path
from typing import Tuple, Dict

import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights
from torch.optim import Adam
from torch.utils.tensorboard import SummaryWriter

# Metrics
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim

# Optional FID (torchmetrics). We'll try to import and use it if available; otherwise skip FID.
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    TORCHMETRICS_FID_AVAILABLE = True
except Exception:
    FrechetInceptionDistance = None
    TORCHMETRICS_FID_AVAILABLE = False

2025-12-02 00:30:40.863236: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# -----------------------------
# Config / Hyperparameters
# -----------------------------
DATA_DIR = r'/home/mmanani/ImageColourizationDataSet/FLAT_TRAIN'   # flat folder with COCO-like images
CHECKPOINT_DIR = r'/home/mmanani/mtechpracticals/semester-4/FullBlownTrainingwithCOCODataset/checkpoints'
GENERATED_DIR = r'/home/mmanani/mtechpracticals/semester-4/FullBlownTrainingwithCOCODataset/generatedcolouredimages'
LOG_DIR = 'runs/colorization_imgnet_contd'



# Basic training params
IMAGE_SIZE = 256          # scaled-up size (power of two). Keep 128/256/512 as needed.
BATCH_SIZE = 16           # lower for higher res; increase if you have GPU memory
NUM_WORKERS = 8
PRETRAIN_EPOCHS = 6
GAN_EPOCHS = 30
LR_G = 4e-5 #Earlier-> 2e-4
LR_D = 2e-5 #Earlier -> 1e-4
BETA1, BETA2 = 0.5, 0.999

LAMBDA_L1 = 20.0          # reduced per our winning settings
LAMBDA_VGG = 0.4
LAMBDA_GP = 10.0
LAMBDA_TV = 0.001

USE_AMP = False           # True can reduce memory but be cautious with GP
N_CRITIC = 5              # recommended for WGAN-GP

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(GENERATED_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


In [3]:
# -----------------------------
# Determinism / seeds
# -----------------------------
# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# -----------------------------
# Helper conversions: LAB <-> RGB (consistent, production-safe)
# -----------------------------
def rgb_np_to_lab_true(rgb_uint8: np.ndarray) -> np.ndarray:
    """uint8 RGB -> LAB with true L scale [0,100]."""
    lab = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2LAB).astype(np.float32)
    # convert OpenCV L(0..255) -> true L(0..100)
    lab[...,0] = lab[...,0] * (100.0 / 255.0)
    return lab

def lab_true_to_rgb_np(lab_true: np.ndarray) -> np.ndarray:
    """lab_true with L in [0,100], a,b in OpenCV scale -> uint8 RGB."""
    lab = lab_true.copy().astype(np.float32)
    lab[...,0] = lab[...,0] * (255.0 / 100.0)
    rgb = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2RGB)
    return rgb

def lab_to_rgb_torch(L: torch.Tensor, ab: torch.Tensor) -> torch.Tensor:
    """
    Differentiable LAB->RGB conversion.
    L: (B,1,H,W) in [-1,1] -> [0,100]
    ab: (B,2,H,W) in [-1,1] -> a,b in approx [-128,128]
    returns: rgb (B,3,H,W) in [0,1]
    """
    L_true = (L + 1.0) * 50.0
    a = ab[:,0:1,:,:] * 128.0
    b = ab[:,1:2,:,:] * 128.0
    lab = torch.cat([L_true, a, b], dim=1).permute(0,2,3,1)  # B,H,W,3

    eps = 0.008856
    kappa = 903.3
    fy = (lab[...,0] + 16.0) / 116.0
    fx = fy + (lab[...,1] / 500.0)
    fz = fy - (lab[...,2] / 200.0)

    def f_to_xyz(f):
        f3 = f ** 3
        return torch.where(f3 > eps, f3, (116.0 * f - 16.0) / kappa)

    Xn, Yn, Zn = 0.95047, 1.0, 1.08883
    X = f_to_xyz(fx) * Xn
    Y = f_to_xyz(fy) * Yn
    Z = f_to_xyz(fz) * Zn

    xyz = torch.stack([X, Y, Z], dim=-1)  # B,H,W,3

    M = torch.tensor([[3.2406, -1.5372, -0.4986],
                      [-0.9689, 1.8758, 0.0415],
                      [0.0557, -0.2040, 1.0570]], device=lab.device, dtype=lab.dtype)
    rgb_lin = torch.tensordot(xyz, M.T, dims=([-1],[0]))  # B,H,W,3
    rgb_lin = rgb_lin.clamp(min=0.0)
    threshold = 0.0031308
    rgb = torch.where(rgb_lin > threshold, 1.055 * (rgb_lin ** (1.0/2.4)) - 0.055, 12.92 * rgb_lin)
    rgb = rgb.permute(0,3,1,2).clamp(0.0, 1.0)
    return rgb.contiguous()

def colorize_external_images(G, DEVICE, tb_writer, epoch, save_dir, image_size=256):
    """
    Colorizes multiple external grayscale images at the end of each epoch.
    Safely handles any image size by resizing to model input resolution.
    
    Args:
        G          : Generator model
        DEVICE     : torch device
        tb_writer  : TensorBoard writer
        epoch      : current epoch index (int)
        save_dir   : directory where generated images will be saved
        image_size : model input resolution (e.g., 256)
    """

    external_paths = [
        "/home/mmanani/mtechpracticals/semester-4/FullBlownTrainingwithCOCODataset/sourceimages/Grayscaleimage43180.jpg",
        "/home/mmanani/mtechpracticals/semester-4/FullBlownTrainingwithCOCODataset/sourceimages/63775aec-17ca-4b4b-a147-fc0d3d426f91.jpg",
        "/home/mmanani/mtechpracticals/semester-4/FullBlownTrainingwithCOCODataset/sourceimages/3ab16d8c-cb90-46f1-b87b-9630c9b2aba8.jpg"
    ]

    G.eval()
    os.makedirs(save_dir, exist_ok=True)

    for idx, external_path in enumerate(external_paths):

        try:
            img_bgr = cv2.imread(external_path)
            if img_bgr is None:
                print(f"[External] Could not load: {external_path}")
                continue

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            # Convert to LAB
            lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)

            # Correct OpenCV L scaling (0–255 → 0–100)
            lab[:, :, 0] *= (100.0 / 255.0)

            # Resize L to model input size
            L_resized = cv2.resize(lab[:, :, 0], (image_size, image_size), interpolation=cv2.INTER_AREA)

            # Normalize
            L_in = ((L_resized / 50.0) - 1.0).astype(np.float32)
            L_in = torch.from_numpy(L_in).unsqueeze(0).unsqueeze(0).to(DEVICE)

            # ---- Generate fake ab ----
            with torch.no_grad():
                pred_ab = G(L_in)
                pred_rgb_tensor = lab_to_rgb_torch(L_in, pred_ab)[0]  # (3,H,W)

            # Convert tensor → uint8 RGB
            pred_rgb = pred_rgb_tensor.permute(1, 2, 0).cpu().numpy()
            pred_uint8 = (np.clip(pred_rgb, 0, 1) * 255).astype(np.uint8)

            # Save output
            out_path = os.path.join(save_dir, f"epoch_{epoch:03d}_external_{idx+1}.png")
            cv2.imwrite(out_path, cv2.cvtColor(pred_uint8, cv2.COLOR_RGB2BGR))

            # Log to TensorBoard under separate tags
            tb_writer.add_image(
                f"GAN/External_Image_{idx+1}",
                torch.tensor(pred_uint8).permute(2, 0, 1),
                epoch
            )

        except Exception as e:
            print(f"[Warning] Failed to colorize {external_path}: {e}")

In [5]:
# -----------------------------
# Dataset (COCO-style flat folder)
# - Resize to IMAGE_SIZE, augment moderately
# -----------------------------
class ColorizationDataset(Dataset):
    def __init__(self, image_dir: str, transform_prob=0.8, max_size_bytes: int = 120 * 1024**3):
        self.image_dir = Path(image_dir)
        if not self.image_dir.exists():
            raise ValueError(f"No such directory: {image_dir}")
        all_files = [p for p in sorted(self.image_dir.iterdir()) if p.suffix.lower() in ('.jpg','.jpeg','.png')]
        if not all_files:
            raise ValueError("No images found in dataset directory.")
        random.shuffle(all_files)
        # optionally cap dataset size to save memory on very large drives (set big default)
        self.image_paths = [str(p) for p in all_files]
        self.transform_prob = transform_prob
        self.augment = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        ])

    def __len__(self):
        return len(self.image_paths)

    def _read_image(self, path):
        try:
            img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
            if img_bgr is None:
                raise Exception("cv2 failed")
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            return img_rgb
        except Exception:
            try:
                pil = Image.open(path).convert('RGB')
                return np.array(pil)
            except Exception:
                return None

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = self._read_image(path)
        if img is None:
            # fallback random
            return self.__getitem__(random.randint(0, len(self.image_paths)-1))

        # Resize to IMAGE_SIZE
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

        # Augment occasionally
        if random.random() < self.transform_prob:
            img = self.augment(transforms.ToPILImage()(img))
            img = np.array(img)

        lab = rgb_np_to_lab_true(img)  # float32 L in [0,100]
        L = (lab[...,0] / 50.0) - 1.0           # [-1,1]
        ab = (lab[...,1:] - 128.0) / 128.0     # [-1,1]
        L_t = torch.from_numpy(np.ascontiguousarray(L)).float().unsqueeze(0)
        ab_t = torch.from_numpy(np.ascontiguousarray(ab.transpose(2,0,1))).float()
        return {'L': L_t, 'ab': ab_t, 'path': path}


In [6]:
# -----------------------------
# Data loaders
# -----------------------------
dataset = ColorizationDataset(DATA_DIR, transform_prob=0.8)
train_size = int(0.95 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
# Print dataset stats
print(f"Total images loaded: {len(dataset)}")

Total images loaded: 118288


In [7]:
# -----------------------------
# Hybrid model classes (final winning)
# -----------------------------
class SelfAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.query = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.key   = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        B, C, H, W = x.size()
        proj_query = self.query(x).view(B, -1, H * W).permute(0, 2, 1)
        proj_key   = self.key(x).view(B, -1, H * W)
        attention  = torch.bmm(proj_query, proj_key).softmax(dim=-1)
        proj_value = self.value(x).view(B, C, H * W)
        out = torch.bmm(proj_value, attention.permute(0, 2, 1)).view(B, C, H, W)
        return self.gamma * out + x

class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(channels),
        )
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, input_channels=1, output_channels=2, base=64):
        super().__init__()
        # Encoder (keeps InstanceNorm)
        self.enc1 = self.down_block(input_channels, base, norm=False)     # IMAGE_SIZE/2
        self.enc2 = self.down_block(base, base*2)                         # /4
        self.enc3 = self.down_block(base*2, base*4)                       # /8
        self.enc4 = self.down_block(base*4, base*8)                       # /16
        # For IMAGE_SIZE=256 -> 256/16 = 16 spatial at bottleneck

        # Bottleneck
        self.res = nn.Sequential(
            ResBlock(base*8),
            SelfAttention(base*8),
            ResBlock(base*8)
        )

        # Decoder (BatchNorm in decoder)
        self.dec1 = self.up_block(base*8, base*4)
        self.dec2 = self.up_block(base*8, base*2)
        self.dec3 = self.up_block(base*4, base)
        self.dec4 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(base*2, output_channels, 3, 1, 1),
            nn.Tanh()
        )

    def down_block(self, in_c, out_c, norm=True):
        layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
        if norm:
            layers.append(nn.InstanceNorm2d(out_c))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        return nn.Sequential(*layers)

    def up_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_c),   # critical for color diversity
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.res(e4)
        d1 = self.dec1(b); d1 = torch.cat([d1, e3], dim=1)
        d2 = self.dec2(d1); d2 = torch.cat([d2, e2], dim=1)
        d3 = self.dec3(d2); d3 = torch.cat([d3, e1], dim=1)
        d4 = self.dec4(d3)
        return d4

class Critic(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.model = nn.Sequential(
            nn.utils.spectral_norm(nn.Conv2d(in_channels, 64, 4, 2, 1)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.utils.spectral_norm(nn.Conv2d(64, 128, 4, 2, 1)),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.utils.spectral_norm(nn.Conv2d(128, 256, 4, 2, 1)),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.utils.spectral_norm(nn.Conv2d(256, 512, 4, 1, 1)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 1)
        )
    def forward(self, x):
        return self.model(x)

In [8]:
# weights init
def weights_init_normal(m):
    classname = m.__class__.__name__
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        try:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        except Exception:
            pass
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)):
        if hasattr(m, 'weight') and m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)

In [9]:
# -----------------------------
# Initialize models, optimizers, perceptual net
# -----------------------------
G = Generator().to(DEVICE)
D = Critic().to(DEVICE)
G.apply(weights_init_normal)
D.apply(weights_init_normal)

optimizer_G = Adam(G.parameters(), lr=LR_G, betas=(BETA1, BETA2))
optimizer_D = Adam(D.parameters(), lr=LR_D, betas=(BETA1, BETA2))

vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:16].to(DEVICE).eval()
for p in vgg.parameters():
    p.requires_grad = False
vgg_mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1,3,1,1)
vgg_std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1,3,1,1)
vgg_loss_fn = nn.MSELoss()

fid_metric = FrechetInceptionDistance(feature=2048).to(DEVICE) if TORCHMETRICS_FID_AVAILABLE else None

In [10]:
# -----------------------------
# Loss helpers & utilities
# -----------------------------
def tv_loss(ab):
    h_diff = torch.abs(ab[:, :, :, 1:] - ab[:, :, :, :-1])
    v_diff = torch.abs(ab[:, :, 1:, :] - ab[:, :, :-1, :])
    return (h_diff.mean() + v_diff.mean())

def compute_gradient_penalty(D, real_samples, fake_samples, device):
    batch_size = real_samples.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    d_interpolates = D(interpolates)
    fake = torch.ones_like(d_interpolates, device=device)
    gradients = torch.autograd.grad(outputs=d_interpolates, inputs=interpolates,
                                    grad_outputs=fake, create_graph=True, retain_graph=True)[0]
    gradients = gradients.view(batch_size, -1)
    gp = ((gradients.norm(2, dim=1) - 1.0) ** 2).mean()
    return gp

# checkpoint helpers
def save_checkpoint(path: str, epoch: int, batch: int, models: Dict[str, nn.Module], optimizers: Dict[str, torch.optim.Optimizer], loss_dict: Dict = None):
    payload = {
        'epoch': epoch,
        'batch': batch,
        'model_state_dict': {k: v.state_dict() for k, v in models.items()},
        'optimizer_state_dict': {k: v.state_dict() for k, v in optimizers.items()},
        'loss': loss_dict
    }
    torch.save(payload, path)

def load_checkpoint(path: str, models: Dict[str, nn.Module], optimizers: Dict[str, torch.optim.Optimizer], map_location: torch.device = DEVICE):
    if not os.path.exists(path):
        return 0, 0
    ckpt = torch.load(path, map_location=map_location)
    msd = ckpt.get('model_state_dict', {})
    osd = ckpt.get('optimizer_state_dict', {})
    if isinstance(msd, dict):
        for k in msd:
            if k in models:
                try:
                    models[k].load_state_dict(msd[k])
                except Exception as e:
                    print(f"Warning loading model {k}: {e}")
    else:
        first = list(models.keys())[0]
        models[first].load_state_dict(msd)
    if isinstance(osd, dict):
        for k in osd:
            if k in optimizers:
                try:
                    optimizers[k].load_state_dict(osd[k])
                except Exception as e:
                    print(f"Warning loading optimizer {k}: {e}")
    else:
        first = list(optimizers.keys())[0]
        optimizers[first].load_state_dict(osd)
    return ckpt.get('epoch', 0), ckpt.get('batch', 0)


In [11]:
# -----------------------------
# Logging and metrics
# -----------------------------
tb_writer = SummaryWriter(LOG_DIR)
CSV_FILE = os.path.join(CHECKPOINT_DIR, 'training_metrics.csv')
if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Mode','Epoch','Batch','G_Loss','D_Loss','PSNR','SSIM','FID'])

In [12]:
# -----------------------------
# Pretraining of Generator (L1 + Perceptual)
# -----------------------------
pretrain_ckpt = os.path.join(CHECKPOINT_DIR, 'pretrain_checkpoint.pth')
start_epoch_pretrain, start_batch_pretrain = 0, 0
if os.path.exists(pretrain_ckpt):
    start_epoch_pretrain, start_batch_pretrain = load_checkpoint(pretrain_ckpt, {'G': G}, {'G': optimizer_G})
    print("Resuming pretrain from:", start_epoch_pretrain, start_batch_pretrain)

scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)
for epoch in range(start_epoch_pretrain, PRETRAIN_EPOCHS):
    G.train()
    running_loss = 0.0
    batches = iter(train_loader)
    if epoch == start_epoch_pretrain and start_batch_pretrain > 0:
        for _ in range(start_batch_pretrain):
            next(batches, None)
    for batch_idx, batch in enumerate(tqdm(batches, desc=f"Pretrain {epoch+1}/{PRETRAIN_EPOCHS}", total=len(train_loader))):
        if epoch == start_epoch_pretrain and batch_idx < start_batch_pretrain:
            continue
        L = batch['L'].to(DEVICE, non_blocking=True)
        real_ab = batch['ab'].to(DEVICE, non_blocking=True)
        optimizer_G.zero_grad(set_to_none=True)
        if USE_AMP:
            with torch.cuda.amp.autocast():
                pred_ab = G(L)
                loss_l1 = F.l1_loss(pred_ab, real_ab) * LAMBDA_L1
                pred_rgb = lab_to_rgb_torch(L, pred_ab)
                real_rgb = lab_to_rgb_torch(L, real_ab)
                pred_norm = (pred_rgb - vgg_mean) / vgg_std
                real_norm = (real_rgb - vgg_mean) / vgg_std
                loss_perc = vgg_loss_fn(vgg(pred_norm), vgg(real_norm)) * LAMBDA_VGG
                loss = loss_l1 + loss_perc
            scaler_G.scale(loss).backward()
            scaler_G.step(optimizer_G)
            scaler_G.update()
        else:
            pred_ab = G(L)
            loss_l1 = F.l1_loss(pred_ab, real_ab) * LAMBDA_L1
            pred_rgb = lab_to_rgb_torch(L, pred_ab)
            real_rgb = lab_to_rgb_torch(L, real_ab)
            pred_norm = (pred_rgb - vgg_mean) / vgg_std
            real_norm = (real_rgb - vgg_mean) / vgg_std
            loss_perc = vgg_loss_fn(vgg(pred_norm), vgg(real_norm)) * LAMBDA_VGG
            loss = loss_l1 + loss_perc
            loss.backward()
            optimizer_G.step()
        running_loss += float(loss.item())
        step = epoch * len(train_loader) + batch_idx
        tb_writer.add_scalar('Pretrain/Total_G_Loss', float(loss.item()), step)
        tb_writer.add_scalar('Pretrain/L1', float(loss_l1.item()), step)
        tb_writer.add_scalar('Pretrain/Perceptual', float(loss_perc.item()), step)
        if (batch_idx + 1) % 1000 == 0:
            save_checkpoint(pretrain_ckpt, epoch, batch_idx+1, {'G': G}, {'G': optimizer_G}, {'loss': float(loss.item())})
    avg_loss = running_loss / len(train_loader)
    print(f"Pretrain Epoch {epoch+1}/{PRETRAIN_EPOCHS} Avg Loss: {avg_loss:.4f}")
    save_checkpoint(pretrain_ckpt, epoch+1, 0, {'G': G}, {'G': optimizer_G}, {'loss': avg_loss})
    # sample to TB
    G.eval()
    with torch.no_grad():
        samp = next(iter(val_loader))
        Ls = samp['L'][:4].to(DEVICE)
        real_abs = samp['ab'][:4].to(DEVICE)
        preds = G(Ls)
        tb_writer.add_images('Pretrain/Pred_RGB', lab_to_rgb_torch(Ls, preds), epoch+1, dataformats='NCHW')
        tb_writer.add_images('Pretrain/Real_RGB', lab_to_rgb_torch(Ls, real_abs), epoch+1, dataformats='NCHW')

print("Pretraining done.")


Resuming pretrain from: 6 0
Pretraining done.


/tmp/ipykernel_4433/119812298.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [13]:
# -----------------------------
# GAN training (WGAN-GP)
# -----------------------------

gan_ckpt_latest = os.path.join(CHECKPOINT_DIR, 'gan_checkpoint.pth')
start_epoch_gan, start_batch_gan = 0, 0
if os.path.exists(gan_ckpt_latest):
    start_epoch_gan, start_batch_gan = load_checkpoint(gan_ckpt_latest, {'G': G, 'D': D}, {'G': optimizer_G, 'D': optimizer_D})
    print("Resuming GAN training:", start_epoch_gan, start_batch_gan)
else:
    # load pretrain generator if available
    if os.path.exists(pretrain_ckpt):
        ck = torch.load(pretrain_ckpt, map_location=DEVICE)
        msd = ck.get('model_state_dict', {})
        if isinstance(msd, dict) and 'G' in msd:
            try:
                G.load_state_dict(msd['G'])
                print("Loaded pretrain G weights.")
            except Exception:
                try:
                    G.load_state_dict(msd)
                except Exception as e:
                    print("Pretrain load failed:", e)

scaler_D = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)

for epoch in range(start_epoch_gan, GAN_EPOCHS):
    G.train(); D.train()
    running_g = 0.0; running_d = 0.0
    metrics_rows = []
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"GAN Epoch {epoch+1}/{GAN_EPOCHS}")
    for batch_idx, batch in pbar:
        if epoch == start_epoch_gan and batch_idx < start_batch_gan:
            continue
        L = batch['L'].to(DEVICE, non_blocking=True)
        real_ab = batch['ab'].to(DEVICE, non_blocking=True)

        # Train critic N_CRITIC times
        d_loss_total = None
        for _ in range(N_CRITIC):
            optimizer_D.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                real_input = torch.cat([L, real_ab], dim=1)
                fake_ab_det = G(L).detach()
                fake_input = torch.cat([L, fake_ab_det], dim=1)
                d_real = D(real_input); d_fake = D(fake_input)
                d_loss = d_fake.mean() - d_real.mean()
            gp = compute_gradient_penalty(D, real_input, fake_input, DEVICE)
            d_loss_total = d_loss + LAMBDA_GP * gp
            if USE_AMP:
                scaler_D.scale(d_loss_total).backward()
                scaler_D.step(optimizer_D)
                scaler_D.update()
            else:
                d_loss_total.backward()
                optimizer_D.step()

        # Train generator once
        optimizer_G.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            fake_ab = G(L)
            fake_ab_clamped = torch.tanh(fake_ab)
            fake_input = torch.cat([L, fake_ab_clamped], dim=1)
            g_adv = -D(fake_input).mean()
            g_l1 = F.l1_loss(fake_ab_clamped, real_ab) * LAMBDA_L1
            pred_rgb = lab_to_rgb_torch(L, fake_ab_clamped)
            real_rgb = lab_to_rgb_torch(L, real_ab)
            pred_norm = (pred_rgb - vgg_mean) / vgg_std
            real_norm = (real_rgb - vgg_mean) / vgg_std
            perc = vgg_loss_fn(vgg(pred_norm), vgg(real_norm)) * LAMBDA_VGG
            g_tv = tv_loss(fake_ab_clamped) * LAMBDA_TV
            loss_G_total = g_adv + g_l1 + perc + g_tv
        if USE_AMP:
            scaler_G.scale(loss_G_total).backward()
            scaler_G.step(optimizer_G)
            scaler_G.update()
        else:
            loss_G_total.backward()
            optimizer_G.step()

        running_d += float(d_loss_total.item()) if d_loss_total is not None else 0.0
        running_g += float(loss_G_total.item())

        # metrics for first sample
        with torch.no_grad():
            fake_rgb = lab_to_rgb_torch(L[:1], fake_ab_clamped[:1]).cpu().permute(0,2,3,1).numpy()[0]
            real_rgb_np = lab_to_rgb_torch(L[:1], real_ab[:1]).cpu().permute(0,2,3,1).numpy()[0]
            fake_uint8 = (np.clip(fake_rgb,0,1)*255).astype(np.uint8)
            real_uint8 = (np.clip(real_rgb_np,0,1)*255).astype(np.uint8)
            fake_t = torch.tensor(fake_uint8).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
            real_t = torch.tensor(real_uint8).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

            fid_metric.update(real_t, real=True)
            fid_metric.update(fake_t, real=False)

            try:
                from skimage.metrics import peak_signal_noise_ratio as sk_psnr
                from skimage.metrics import structural_similarity as sk_ssim
                psnr_val = float(sk_psnr(real_uint8, fake_uint8, data_range=255))
                ssim_val = float(sk_ssim(real_uint8, fake_uint8, channel_axis=-1, data_range=255, win_size=3))
            except Exception:
                psnr_val = float('nan'); ssim_val = float('nan')

        # FID update best-effort
        fid_value = None
        if fid_metric is not None:
            try:
                fake_t = torch.from_numpy(fake_rgb.transpose(2,0,1)[None,:,:,:].astype(np.float32)).to(DEVICE)
                real_t = torch.from_numpy(real_rgb_np.transpose(2,0,1)[None,:,:,:].astype(np.float32)).to(DEVICE)
                fid_metric.update(fake_t, real=False)
                fid_metric.update(real_t, real=True)
                fid_value = float(fid_metric.compute().cpu().item()); fid_metric.reset()
            except Exception:
                fid_value = None

        step = epoch * len(train_loader) + batch_idx
        tb_writer.add_scalar('GAN/Generator_total', float(loss_G_total.item()), step)
        tb_writer.add_scalar('GAN/Generator_adv', float(g_adv.item()), step)
        tb_writer.add_scalar('GAN/Generator_L1', float(g_l1.item()), step)
        tb_writer.add_scalar('GAN/Generator_perceptual', float(perc.item()), step)
        tb_writer.add_scalar('GAN/Discriminator', float(d_loss_total.item()), step)
        tb_writer.add_scalar('Metrics/PSNR', psnr_val, step)
        tb_writer.add_scalar('Metrics/SSIM', ssim_val, step)
        metrics_rows.append(['train', epoch+1, batch_idx,float(loss_G_total.item()),float(d_loss_total.item()),psnr_val, ssim_val])
        if (batch_idx + 1) % 1000 == 0:
            save_checkpoint(os.path.join(CHECKPOINT_DIR, 'gan_checkpoint.pth'), epoch+1, batch_idx+1, {'G': G, 'D': D}, {'G': optimizer_G, 'D': optimizer_D}, {'g_loss': float(loss_G_total.item()), 'd_loss': float(d_loss_total.item())})
            try:
                cv2.imwrite(os.path.join(GENERATED_DIR, f'ep{epoch+1}_b{batch_idx+1}.png'), cv2.cvtColor(fake_uint8, cv2.COLOR_RGB2BGR))
            except Exception:
                pass

        if batch_idx % 50 == 0:
            torch.cuda.empty_cache(); gc.collect()
        pbar.set_postfix({'G': f'{float(loss_G_total.item()):.4f}', 'D': f'{float(d_loss_total.item()):.4f}'})

    # epoch end
    with open(CSV_FILE, 'a', newline='') as f:
        writer = csv.writer(f); writer.writerows(metrics_rows)

    avg_g = running_g / len(train_loader); avg_d = running_d / len(train_loader)
    # -----------------------------
    # Compute FID once per epoch
    # -----------------------------
    fid_value = None
    if fid_metric is not None:
        try:
            fid_value = float(fid_metric.compute().cpu().item())
        except Exception:
            fid_value = None

        fid_metric.reset()  # reset after epoch
    print(f"Epoch {epoch+1}/{GAN_EPOCHS} | G_loss: {avg_g:.4f} | D_loss: {avg_d:.4f} | FID_Value:{fid_value}")
    # Log epoch-level FID
    tb_writer.add_scalar("Metrics/FID", 0 if fid_value is None else fid_value, epoch+1)

    # save epoch checkpoint both epoch-specific and rolling latest
    epoch_path = os.path.join(CHECKPOINT_DIR, f'gan_checkpoint_epoch_{epoch+1:03d}.pth')
    save_checkpoint(epoch_path, epoch+1, 0, {'G': G, 'D': D}, {'G': optimizer_G, 'D': optimizer_D}, {'g_loss': avg_g, 'd_loss': avg_d})
    save_checkpoint(os.path.join(CHECKPOINT_DIR, 'gan_checkpoint.pth'), epoch+1, 0, {'G': G, 'D': D}, {'G': optimizer_G, 'D': optimizer_D}, {'g_loss': avg_g, 'd_loss': avg_d})
    colorize_external_images(G, DEVICE, tb_writer, epoch+1,save_dir=GENERATED_DIR, image_size=IMAGE_SIZE)
    # validation sample logging
    G.eval()
    with torch.no_grad():
        sample = next(iter(val_loader))
        Ls = sample['L'][:4].to(DEVICE)
        real_abs = sample['ab'][:4].to(DEVICE)
        preds = G(Ls)
        tb_writer.add_images('GAN/Pred_RGB', lab_to_rgb_torch(Ls, preds), epoch+1, dataformats='NCHW')
        tb_writer.add_images('GAN/Real_RGB', lab_to_rgb_torch(Ls, real_abs), epoch+1, dataformats='NCHW')
        first_rgb = (lab_to_rgb_torch(Ls, preds)[0].permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
        cv2.imwrite(os.path.join(GENERATED_DIR, f'epoch_{epoch+1:03d}_sample.png'), cv2.cvtColor(first_rgb, cv2.COLOR_RGB2BGR))
    G.train()

# cleanup and final save
tb_writer.close()
torch.save(G.state_dict(), os.path.join(CHECKPOINT_DIR, 'generator_final.pth'))
torch.save(D.state_dict(), os.path.join(CHECKPOINT_DIR, 'critic_final.pth'))
print("Training finished. Models saved.")


Resuming GAN training: 6 0


/tmp/ipykernel_4433/2032799068.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_D = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_4433/2032799068.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)
GAN Epoch 7/30:   0%|          | 0/7024 [00:00<?, ?it/s]/tmp/ipykernel_4433/2032799068.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_4433/2032799068.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
GAN Epoch 7/30: 100%|██████████| 7024/7024 [3:43:46<00:00,  1.91s/it, G=2.6624, D=

Epoch 7/30 | G_loss: 2.6540 | D_loss: -0.0222 | FID_Value:18.04758644104004


GAN Epoch 8/30: 100%|██████████| 7024/7024 [3:32:58<00:00,  1.82s/it, G=2.0885, D=0.0425]   


Epoch 8/30 | G_loss: 2.6834 | D_loss: -0.0225 | FID_Value:18.547800064086914


...

Epoch 28/30 | G_loss: 2.6829 | D_loss: -0.0226 | FID_Value:16.236896514892578


GAN Epoch 29/30: 100%|██████████| 7024/7024 [3:46:23<00:00,  1.93s/it, G=2.5186, D=0.0330]   


Epoch 29/30 | G_loss: 2.6807 | D_loss: -0.0226 | FID_Value:16.037864685058594


GAN Epoch 30/30: 100%|██████████| 7024/7024 [3:50:36<00:00,  1.97s/it, G=2.9154, D=-0.0133]  


Epoch 30/30 | G_loss: 2.6798 | D_loss: -0.0225 | FID_Value:15.955697059631348
Training finished. Models saved.
